# 本章总结

本章围绕 PyTorch 神经网络基础展开，重点理解 `nn.Module` 是构建模型的核心抽象：层、块以及完整网络都可以看作 Module 的子类。通过 `nn.Sequential` 可以快速串联网络层；通过自定义类并实现 `__init__` 与 `forward`，可以灵活定义网络结构和前向传播逻辑。

主要内容包括：

- **层和块**：使用 `nn.Sequential` 快速搭建多层感知机，理解网络由多个层/块组合而成。
- **自定义块**：继承 `nn.Module`，在 `__init__` 中声明层，在 `forward` 中定义计算过程。
- **顺序块与嵌套块**：手写类似 Sequential 的容器，理解 `_modules` 如何保存子模块，以及如何递归组合复杂网络。
- **正向传播的灵活性**：在 `forward` 中可以加入普通 Python 控制流、矩阵运算、循环和条件逻辑。
- **参数管理**：通过 `state_dict()`、`named_parameters()` 等方式访问权重和偏置，理解参数的形状、数据和梯度。
- **参数初始化与替换**：使用 PyTorch 内置初始化方法，或自定义初始化规则，并能直接修改参数数据。
- **参数绑定**：多个层可以共享同一个参数对象，实现权重共享。
- **自定义层**：自定义无参数层和带参数层，使用 `nn.Parameter` 注册可训练参数。
- **读写文件**：使用 `torch.save` 和 `torch.load` 保存/加载张量、字典以及模型参数。

本章的核心思想是：PyTorch 通过 `nn.Module` 将模型结构、参数管理、前向计算和保存加载统一起来；掌握 Module 的组织方式后，就可以从简单的 Sequential 网络扩展到高度自定义、可嵌套、可共享参数的复杂神经网络。

## 综合实战代码示例

下面的代码把本章知识点集合到一个小任务中：**训练一个神经网络判断二维点是在单位圆内还是单位圆外**。这个例子同时用到了 `nn.Module`、`nn.Sequential`、自定义层、嵌套块、前向传播、参数初始化、参数查看、参数共享以及模型参数保存/加载。

In [58]:
# ===== 本章综合实战代码示例：判断二维点在单位圆内还是圆外 =====
# 问题描述：
# 给定一个二维点 (x1, x2)，判断它是在单位圆内部，还是在单位圆外部。
# 如果 x1^2 + x2^2 <= 1，标签为 0，表示“圆内”；
# 如果 x1^2 + x2^2 > 1，标签为 1，表示“圆外”。
# 这个任务不是简单直线可分的，所以适合用一个小型神经网络来解决。

import torch
from torch import nn
from torch.nn import functional as F

# 固定随机种子，方便每次运行得到相近的结果。
torch.manual_seed(0)

# ------------------------------------------------------------
# 1. 构造一个小数据集
# ------------------------------------------------------------

# 随机生成 512 个二维点。
# torch.rand(512, 2) 的范围是 [0, 1)，乘以 3 再减 1.5 后，范围约为 [-1.5, 1.5)。
X = torch.rand(512, 2) * 3 - 1.5

# 根据点到原点的距离构造标签。
# X[:, 0] 表示所有点的第 1 个坐标，X[:, 1] 表示所有点的第 2 个坐标。
# long() 是因为分类任务的标签通常需要是整数类型。
y = ((X[:, 0] ** 2 + X[:, 1] ** 2) > 1.0).long()

print('输入 X 的形状:', X.shape)      # 512 个样本，每个样本 2 个特征
print('标签 y 的形状:', y.shape)      # 512 个标签
print('前 5 个标签:', y[:5])

# ------------------------------------------------------------
# 2. 自定义一个没有参数的层
# ------------------------------------------------------------

class CenteredLayer(nn.Module):
    """让输入减去当前小批量的均值，使数据更接近以 0 为中心。"""

    def __init__(self):
        super().__init__()

    def forward(self, X):
        # dim=0 表示按“每一列特征”求均值。
        # keepdim=True 可以保留二维形状，方便和 X 广播相减。
        return X - X.mean(dim=0, keepdim=True)

# ------------------------------------------------------------
# 3. 自定义一个特征提取块：里面混合使用 Sequential、普通层、自定义层和参数绑定
# ------------------------------------------------------------

class FeatureBlock(nn.Module):
    def __init__(self, in_units, hidden_units):
        super().__init__()

        # 第一层：把输入特征映射到隐藏维度。
        self.input = nn.Linear(in_units, hidden_units)

        # shared 是一个共享层，后面会在 Sequential 中使用两次。
        # 同一个层对象用两次，就表示这两处共享同一组权重和偏置。
        self.shared = nn.Linear(hidden_units, hidden_units)

        # 使用 nn.Sequential 组织一个顺序执行的子网络。
        self.net = nn.Sequential(
            self.input,
            nn.ReLU(),
            self.shared,
            nn.ReLU(),
            self.shared,      # 再次使用同一个 shared 层：这就是参数绑定
            nn.ReLU(),
            CenteredLayer(),  # 自定义无参数层
        )

    def forward(self, X):
        # forward 中定义数据如何流过这个块。
        return self.net(X)

# ------------------------------------------------------------
# 4. 自定义完整模型
# ------------------------------------------------------------

class CircleClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        # 嵌套使用自定义块。
        self.features = FeatureBlock(in_units=2, hidden_units=16)

        # 输出层：输出 2 个分数，分别对应“圆内”和“圆外”。
        self.output = nn.Linear(16, 2)

    def forward(self, X):
        # 先提取特征，再输出分类分数。
        # CrossEntropyLoss 会直接接收这种未经过 softmax 的原始分数 logits。
        return self.output(self.features(X))

net = CircleClassifier()

# ------------------------------------------------------------
# 5. 参数初始化
# ------------------------------------------------------------

def init_weights(m):
    # isinstance(m, nn.Linear) 表示只初始化线性层。
    if isinstance(m, nn.Linear):
        # ReLU 网络常用 Kaiming/He 初始化。
        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
        nn.init.zeros_(m.bias)

# apply 会递归访问 net 内部所有子模块，并把 init_weights 应用到它们身上。
net.apply(init_weights)

# ------------------------------------------------------------
# 6. 查看参数名称和形状
# ------------------------------------------------------------

print()
print('模型参数名称和形状：')
for name, param in net.named_parameters():
    print(name, param.shape)

# 验证参数绑定：net.features.net[2] 和 net.features.net[4] 是同一个 shared 层。
print()
print('shared 层是否被重复使用:', net.features.net[2] is net.features.net[4])

# ------------------------------------------------------------
# 7. 训练模型
# ------------------------------------------------------------

loss_fn = nn.CrossEntropyLoss()                 # 多分类交叉熵损失
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)  # 随机梯度下降优化器

for epoch in range(400):
    
    # 清空上一轮的梯度。
    optimizer.zero_grad()

    # 前向传播：得到每个样本属于 2 个类别的分数。
    logits = net(X)

    # 计算损失。
    loss = loss_fn(logits, y)

    # 反向传播：计算每个参数的梯度。
    loss.backward()

    # 根据梯度更新参数。
    optimizer.step()

    # 每隔 50 轮打印一次训练情况。
    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            pred = net(X).argmax(dim=1)
            acc = (pred == y).float().mean()
        print(f'epoch {epoch + 1:3d}, loss={loss.item():.4f}, acc={acc.item():.4f}')

# ------------------------------------------------------------
# 8. 用训练好的模型做验证/预测
# ------------------------------------------------------------
# 不只测试 4 个点，而是准备更多验证点，覆盖：
# 1. 明显在圆内的点；
# 2. 接近单位圆边界的点；
# 3. 明显在圆外的点；
# 4. 随机生成的一批新点。

manual_test_X = torch.tensor([
    [0.0, 0.0],     # 圆心：圆内
    [0.3, 0.4],     # 0.3^2 + 0.4^2 = 0.25：圆内
    [-0.5, 0.5],    # 0.25 + 0.25 = 0.5：圆内
    [0.7, 0.0],     # 0.49：圆内
    [0.99, 0.0],    # 非常接近边界，但仍在圆内
    [0.0, -0.99],   # 非常接近边界，但仍在圆内
    [1.0, 0.0],     # 正好在边界上，按定义属于圆内
    [0.0, 1.0],     # 正好在边界上，按定义属于圆内
    [1.01, 0.0],    # 刚超过边界：圆外
    [0.0, -1.01],   # 刚超过边界：圆外
    [0.8, 0.8],     # 0.64 + 0.64 = 1.28：圆外
    [-0.9, -0.9],   # 0.81 + 0.81 = 1.62：圆外
    [1.2, 0.0],     # 圆外
    [0.0, 1.2],     # 圆外
    [-1.3, 0.4],    # 圆外
    [1.4, -1.4],    # 圆外
])

# 再随机生成 32 个验证点，范围仍然是 [-1.5, 1.5)。
# 这些点没有参与训练，用来观察模型在新点上的表现。
random_test_X = torch.rand(32, 2) * 3 - 1.5

# 把人工设计的点和随机点拼接到一起，形成更丰富的验证集。
test_X = torch.cat([manual_test_X, random_test_X], dim=0)

# 根据真实公式计算验证集标签，用来和模型预测结果对比。
test_y = ((test_X[:, 0] ** 2 + test_X[:, 1] ** 2) > 1.0).long()

with torch.no_grad():
    pred = net(test_X).argmax(dim=1)
    test_acc = (pred == test_y).float().mean()

label_names = ['圆内', '圆外']
print()
print('验证集样本数量:', len(test_X))
print('验证集准确率:', test_acc.item())
print('前 20 个验证样本的预测结果：')

# 只打印前 20 个，避免输出太长；完整准确率已经在上面统计。
for point, true_label, pred_label in zip(test_X[:20], test_y[:20], pred[:20]):
    print(
        f'点 {point.tolist()} | '
        f'真实：{label_names[true_label]} | '
        f'预测：{label_names[pred_label]}'
    )

# ------------------------------------------------------------
# 9. 保存和加载模型参数
# ------------------------------------------------------------

# 保存的不是整个模型对象，而是模型参数字典 state_dict。
torch.save(net.state_dict(), 'circle_classifier.params')

# 要加载参数，必须先创建同样结构的模型。
clone = CircleClassifier()
clone.load_state_dict(torch.load('circle_classifier.params'))
clone.eval()

# 检查加载后的模型和原模型对同一批输入的输出是否一致。
with torch.no_grad():
    same_output = torch.allclose(net(test_X), clone(test_X))

print()
print('保存并重新加载后，输出是否一致:', same_output)

输入 X 的形状: torch.Size([512, 2])
标签 y 的形状: torch.Size([512])
前 5 个标签: tensor([0, 1, 0, 1, 0])

模型参数名称和形状：
features.input.weight torch.Size([16, 2])
features.input.bias torch.Size([16])
features.shared.weight torch.Size([16, 16])
features.shared.bias torch.Size([16])
output.weight torch.Size([2, 16])
output.bias torch.Size([2])

shared 层是否被重复使用: True
epoch  50, loss=0.1551, acc=0.9590
epoch 100, loss=0.1083, acc=0.9688
epoch 150, loss=0.0903, acc=0.9785
epoch 200, loss=0.0796, acc=0.9844
epoch 250, loss=0.0723, acc=0.9863
epoch 300, loss=0.0667, acc=0.9883
epoch 350, loss=0.0623, acc=0.9883
epoch 400, loss=0.0587, acc=0.9902

验证集样本数量: 48
验证集准确率: 0.8125
前 20 个验证样本的预测结果：
点 [0.0, 0.0] | 真实：圆内 | 预测：圆内
点 [0.30000001192092896, 0.4000000059604645] | 真实：圆内 | 预测：圆内
点 [-0.5, 0.5] | 真实：圆内 | 预测：圆内
点 [0.699999988079071, 0.0] | 真实：圆内 | 预测：圆内
点 [0.9900000095367432, 0.0] | 真实：圆内 | 预测：圆外
点 [0.0, -0.9900000095367432] | 真实：圆内 | 预测：圆外
点 [1.0, 0.0] | 真实：圆内 | 预测：圆外
点 [0.0, 1.0] | 真实：圆内 | 预测：圆外
点 [1.00999999046

/tmp/ipykernel_112337/1079074552.py:221: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clone.load_state_dict(torch.load('circle_classifier.params'))


# 1. 层和块

① nn.Sequential 定义了一种特殊的Module。

In [38]:
# 回顾一下多层感知机（MLP, Multilayer Perceptron）
# 这个例子展示：如何用 PyTorch 的 nn.Sequential 快速搭建一个前馈神经网络。

import torch                 # PyTorch 的核心库，提供张量 Tensor、自动求导等功能
from torch import nn          # nn 模块中封装了神经网络常用的层、损失函数、容器等
from torch.nn import functional as F  # functional 中提供函数式 API，例如 relu、softmax 等

# nn.Sequential 会按照传入层的顺序依次执行，适合“从前到后直线型”的网络。
# 输入形状：每个样本有 20 个特征。
# 第一层 Linear(20, 256)：把 20 维输入映射到 256 维隐藏表示。
# ReLU：非线性激活函数，让模型能够表达非线性关系。
# 第二层 Linear(256, 10)：把 256 维隐藏表示映射到 10 维输出，常见于 10 类分类任务。
net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

# 随机生成一个小批量输入 X。
# 形状为 (2, 20)：表示 batch_size=2，即有 2 个样本；每个样本有 20 个特征。
X = torch.rand(2, 20)

# 像调用函数一样调用 net，会自动执行 net.forward(X)。
# 输出形状为 (2, 10)：2 个样本，每个样本得到 10 个输出值。
net(X)

tensor([[ 0.0081, -0.3484,  0.1468, -0.0083,  0.0540,  0.2178, -0.0964,  0.2684,
          0.1011, -0.0673],
        [ 0.1319, -0.1910,  0.0971, -0.0861, -0.1813,  0.2168, -0.0685,  0.1782,
          0.2134, -0.1033]], grad_fn=<AddmmBackward0>)

# 2. 自定义块

* 任何的层或者神经网络都可以认为是Module的子类
* 自定义块的两个关键函数：__init__ 和 forward

In [39]:
# 自定义块：通过继承 nn.Module 来定义自己的神经网络结构。
# 当 Sequential 不够灵活时，就可以自己写一个类来组织层和前向传播逻辑。

import torch
from torch import nn
from torch.nn import functional as F

# MLP 继承自 nn.Module，因此它也是一个 PyTorch 模型/模块。
class MLP(nn.Module):
    def __init__(self):
        # super().__init__() 会初始化父类 nn.Module 内部需要的状态。
        # 自定义 Module 时通常第一行就写它，否则参数可能无法被 PyTorch 正确管理。
        super().__init__()

        # 在 __init__ 中声明网络需要用到的层。
        # hidden 是隐藏层：输入 20 维，输出 256 维。
        self.hidden = nn.Linear(20, 256)

        # out 是输出层：输入 256 维，输出 10 维。
        self.out = nn.Linear(256, 10)

    # forward 定义“输入 X 如何一步步变成输出”。
    # 注意：我们通常不直接调用 forward，而是调用 net(X)，PyTorch 会自动转到 forward。
    def forward(self, X):
        # 计算流程：
        # 1. self.hidden(X)：线性变换，把 20 维变成 256 维。
        # 2. F.relu(...)：加入非线性激活。
        # 3. self.out(...)：输出层，把 256 维变成 10 维。
        return self.out(F.relu(self.hidden(X)))

# 实例化模型。此时 hidden 和 out 两个线性层的参数也会被创建出来。
net = MLP()

# 构造一个形状为 (2, 20) 的输入小批量。
X = torch.rand(2, 20)

# 调用模型进行前向传播，得到形状为 (2, 10) 的输出。
net(X)

tensor([[ 0.2036,  0.0101, -0.1744, -0.2051,  0.0718,  0.0465, -0.1906, -0.1853,
         -0.0400, -0.2078],
        [ 0.1850, -0.1327, -0.1017, -0.0970,  0.0227, -0.0813, -0.0662, -0.2813,
         -0.0181, -0.1426]], grad_fn=<AddmmBackward0>)

# 3. 顺序块

In [40]:
# 手写一个类似 nn.Sequential 的顺序块。
# 目标：理解 PyTorch 的 Module 如何保存子模块，以及 forward 如何依次调用它们。

class MySequential(nn.Module):
    def __init__(self, *args):
        # *args 表示可以接收任意多个位置参数。
        # 例如 MySequential(layer1, layer2, layer3) 中的三个层都会进入 args。
        super().__init__()

        # nn.Module 内部使用 _modules 字典保存子模块。
        # 正式项目中更常用 self.add_module(name, module) 或直接使用 nn.Sequential。
        # 这里直接操作 _modules，是为了帮助理解 Sequential 的底层思想。
        for block in args:
            # 把每一个传入的层/块保存起来，后面 forward 时按顺序取出并执行。
            self._modules[block] = block

    def forward(self, X):
        # self._modules.values() 会依次取出保存的子模块。
        for block in self._modules.values():
            # 打印当前正在执行的层，方便观察数据经过网络的顺序。
            print(block)

            # 把上一层的输出作为下一层的输入。
            X = block(X)

        # 所有层执行完后，返回最终结果。
        return X

# 构造一个三层顺序网络：Linear -> ReLU -> Linear。
net = MySequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

# 输入包含 2 个样本，每个样本 20 个特征。
X = torch.rand(2, 20)

# 执行前向传播，同时会打印每一层的结构。
net(X)

Linear(in_features=20, out_features=256, bias=True)
ReLU()
Linear(in_features=256, out_features=10, bias=True)


tensor([[-2.2963e-01,  1.8281e-01, -1.6899e-01, -6.1744e-02,  2.8249e-01,
          1.3255e-04,  1.7747e-02, -2.0895e-01, -2.2809e-01,  1.5018e-01],
        [-1.4546e-01,  1.1400e-01,  4.1961e-03, -7.5291e-02,  1.4668e-01,
         -3.7974e-02,  1.3292e-01, -2.7853e-01, -2.9535e-01,  2.1980e-01]],
       grad_fn=<AddmmBackward0>)

# 4. 正向传播

In [41]:
# 更灵活的正向传播：在 forward 中可以写普通 Python 代码。
# 这说明 PyTorch 的动态图机制非常灵活，不局限于简单的层叠结构。

class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()

        # rand_weight 是一个固定随机权重矩阵，形状为 (20, 20)。
        # requires_grad=False 表示它不参与梯度计算，也不会在训练中被更新。
        self.rand_weight = torch.rand((20, 20), requires_grad=False)

        # 这个线性层会被调用两次，因此它的参数在两次调用中是共享的。
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        # 第一次线性变换：输入和输出维度都是 20。
        X = self.linear(X)

        # torch.mm 是矩阵乘法。
        # self.rand_weight + 1 只是为了演示可以在 forward 中进行普通张量运算。
        # F.relu 会把小于 0 的元素变成 0，大于 0 的元素保持不变。
        X = F.relu(torch.mm(X, self.rand_weight + 1))

        # 第二次调用同一个 self.linear，因此这里复用了同一组权重和偏置。
        X = self.linear(X)

        # 在 forward 中使用 while 循环：只要张量元素绝对值之和大于 1，就整体除以 2。
        # 这展示了 forward 中可以包含控制流。
        while X.abs().sum() > 1:
            X /= 2

        # 返回一个标量，表示所有元素求和后的结果。
        return X.sum()

# 实例化自定义模型。
net = FixedHiddenMLP()

# 构造输入：2 个样本，每个样本 20 个特征。
X = torch.rand(2, 20)

# 执行自定义的前向传播逻辑。
net(X)

tensor(-0.0689, grad_fn=<SumBackward0>)

# 5. 混合组合块（套娃）

In [42]:
# 混合组合块：把自定义 Module、Sequential 和普通层嵌套组合起来。
# 神经网络可以像“搭积木”一样，由小块组合成大块。

class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()

        # self.net 是一个内部小网络：20 -> 64 -> 32。
        # 它本身是 nn.Sequential，也属于 nn.Module。
        self.net = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # 额外再接一个线性层：32 -> 16。
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        # 先经过内部 Sequential，再经过 self.linear。
        return self.linear(self.net(X))

# chimear 可以理解为一个“混合模型”：
# 1. NestMLP：20 -> 16
# 2. nn.Linear(16, 20)：16 -> 20
# 3. FixedHiddenMLP：输入 20 维，最后输出一个标量
chimear = nn.Sequential(
    NestMLP(),
    nn.Linear(16, 20),
    FixedHiddenMLP()
)

# 输入 2 个样本，每个样本 20 维。
X = torch.rand(2, 20)

# 整个组合模型会按顺序执行三个模块。
chimear(X)

tensor(-0.0286, grad_fn=<SumBackward0>)

# 6. 参数管理

In [43]:
# 参数管理：查看、访问和理解模型中的权重与偏置。
# 首先关注一个具有单隐藏层的多层感知机。

import torch
from torch import nn

# 网络结构：4 -> 8 -> 1。
# net[0] 是第一个 Linear，net[1] 是 ReLU，net[2] 是第二个 Linear。
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

# 输入 X 的形状为 (2, 4)：2 个样本，每个样本 4 个特征。
X = torch.rand(size=(2, 4))

# 前向传播，输出形状为 (2, 1)。
print(net(X))

# state_dict() 返回这一层的参数字典。
# 对 Linear 层来说，通常包含 weight 和 bias。
print(net[2].state_dict())

# bias 是一个 Parameter 对象。
# Parameter 是 Tensor 的特殊子类，会被 PyTorch 识别为可训练参数。
print(type(net[2].bias))
print(net[2].bias)

# .data 可以查看参数内部保存的张量值。
# 初学阶段可以用来观察参数；真实训练中应谨慎直接修改 .data。
print(net[2].bias.data)

# 还没有执行反向传播 backward()，所以权重的梯度 grad 目前是 None。
print(net[2].weight.grad == None)

# named_parameters() 可以遍历某个模块自己的参数名称和形状。
# net[0] 是第一层 Linear，它有 weight 和 bias 两组参数。
print(*[(name, param.shape) for name, param in net[0].named_parameters()])

# 对整个 net 调用 named_parameters()，会递归遍历所有子模块的参数。
# 名字中的 0、2 表示这些参数来自 net[0]、net[2]。
# ReLU 没有可训练参数，因此不会出现在结果里。
print(*[(name, param.shape) for name, param in net.named_parameters()])

# 也可以通过 state_dict 的键名直接访问某个参数。
print(net.state_dict()['2.bias'].data)

tensor([[0.1932],
        [0.2734]], grad_fn=<AddmmBackward0>)
OrderedDict([('weight', tensor([[ 0.1090,  0.0880,  0.2118, -0.2527, -0.0391,  0.1815, -0.1418, -0.0352]])), ('bias', tensor([0.2156]))])
<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.2156], requires_grad=True)
tensor([0.2156])
True
('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))
tensor([0.2156])


# 7. 嵌套块

In [44]:
# 嵌套块中的参数收集。
# 即使模型由很多层嵌套而成，PyTorch 也能递归找到所有子模块和参数。

# block1 返回一个小的 Sequential：4 -> 8 -> 4。
def block1():
    return nn.Sequential(
        nn.Linear(4, 8),
        nn.ReLU(),
        nn.Linear(8, 4),
        nn.ReLU()
    )

# block2 会把 4 个 block1 继续嵌套在一个 Sequential 里。
def block2():
    net = nn.Sequential()

    for i in range(4):
        # add_module(name, module) 用来给子模块起名字并加入当前网络。
        # f'block{i}' 会生成 block0、block1、block2、block3 这样的名字。
        net.add_module(f'block{i}', block1())

    return net

# rgnet 的结构：先经过 block2()，最后接一个 Linear(4, 1)。
rgnet = nn.Sequential(block2(), nn.Linear(4, 1))

# X 来自前面的代码单元，形状为 (2, 4)，与 rgnet 的输入维度匹配。
print(rgnet(X))

# 打印网络结构，观察嵌套模块如何被组织和命名。
print(rgnet)

tensor([[0.5712],
        [0.5712]], grad_fn=<AddmmBackward0>)
Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


# 8 内置初始化

In [45]:
# 内置初始化：用 PyTorch 提供的方法初始化参数。
# 合理的参数初始化可以帮助神经网络更稳定地训练。

net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

# 这个函数会被 apply 传给网络中的每一个子模块。
def init_normal(m):
    # 只对 Linear 层做初始化；ReLU 这类层没有 weight/bias。
    if type(m) == nn.Linear:
        # normal_ 表示原地操作：直接把 m.weight 修改为正态分布随机值。
        # mean=0, std=0.01 表示均值为 0，标准差为 0.01。
        nn.init.normal_(m.weight, mean=0, std=0.01)

        # 把偏置初始化为 0。
        nn.init.zeros_(m.bias)

# apply 会递归访问 net 中的所有子模块，并对每个模块调用 init_normal。
net.apply(init_normal)

# 查看第一层权重矩阵的第一行，以及第一层偏置的第一个元素。
print(net[0].weight.data[0])
print(net[0].bias.data[0])

tensor([ 0.0081, -0.0036, -0.0067,  0.0097])
tensor(0.)


In [46]:
# 常数初始化：把权重全部初始化为固定值。
# 这主要用于演示初始化方法；真实训练中通常不会把所有权重都设成同一个值。

net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

def init_constant(m):
    if type(m) == nn.Linear:
        # 把线性层的所有权重都设置为 1。
        nn.init.constant_(m.weight, 1)

        # 把线性层的所有偏置都设置为 0。
        nn.init.zeros_(m.bias)

# 对网络中的所有 Linear 层应用常数初始化。
net.apply(init_constant)

# 查看第一层初始化后的权重和偏置。
print(net[0].weight.data[0])
print(net[0].bias.data[0])

tensor([1., 1., 1., 1.])
tensor(0.)


In [47]:
# 对不同层使用不同的初始化方法。
# 有时我们希望网络中不同层采用不同的初始化策略。

# Xavier 初始化常用于含有线性层/激活函数的网络，可以帮助控制信号方差。
def xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

# 自定义一个把权重全部设为 42 的初始化函数。
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)  # 这里只是演示“指定层单独初始化”。

# 只对 net[0] 这一层应用 Xavier 初始化。
net[0].apply(xavier)

# 只对 net[2] 这一层应用常数 42 初始化。
net[2].apply(init_42)

# 打印两层的权重，观察它们使用了不同的初始化方式。
print(net[0].weight.data)
print(net[2].weight.data)

tensor([[ 0.0625, -0.0899, -0.6924,  0.5247],
        [-0.1456, -0.0093, -0.4226, -0.0277],
        [ 0.3057, -0.0681,  0.2588, -0.4618],
        [-0.2957,  0.2866, -0.5412,  0.3275],
        [-0.4609, -0.4992,  0.0465,  0.4453],
        [-0.6320, -0.0842,  0.2635,  0.2859],
        [ 0.2886,  0.4517, -0.0548,  0.0731],
        [ 0.6692, -0.6620,  0.0229, -0.5213]])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


# 9. 参数替换

In [48]:
# 自定义初始化与参数替换。
# 这个例子展示：除了调用内置初始化函数，还可以用张量操作手动修改参数。

def my_init(m):
    if type(m) == nn.Linear:
        # 打印当前正在初始化的参数名称和形状。
        # named_parameters() 返回 (参数名, 参数张量)。这里取第一个参数，通常是 weight。
        print("Init", *[(name, param.shape) for name, param in m.named_parameters()][0])

        # 将权重初始化为 [-10, 10] 之间的均匀分布随机数。
        nn.init.uniform_(m.weight, -10, 10)

        # 只保留绝对值大于等于 5 的权重，小于 5 的权重置为 0。
        # m.weight.data.abs() >= 5 会得到一个布尔矩阵。
        # 布尔值参与乘法时 True 相当于 1，False 相当于 0。
        m.weight.data *= m.weight.data.abs() >= 5

# 对网络中所有 Linear 层应用自定义初始化。
net.apply(my_init)

# 查看第一层权重的前两行。
print(net[0].weight[:2])

# 直接修改参数值：把第一层所有权重都加 1。
# 注意：直接操作 .data 适合观察和演示；实际训练代码中要非常谨慎。
net[0].weight.data[:] += 1

# 再把第一层权重矩阵左上角的元素手动改成 42。
net[0].weight.data[0, 0] = 42

# 打印第一层权重的第一行，观察手动替换后的结果。
print(net[0].weight.data[0])

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])
tensor([[ 0.0000,  5.4626,  7.1170,  0.0000],
        [ 0.0000, -6.1083,  9.6116,  7.3448]], grad_fn=<SliceBackward0>)
tensor([42.0000,  6.4626,  8.1170,  1.0000])


# 10. 参数绑定

In [49]:
# 参数绑定：让多个层共享同一组权重。
# 如果同一个 nn.Linear 对象被放到网络的多个位置，它们引用的是同一份参数。

# 先创建一个将 8 维映射到 8 维的线性层。
shared = nn.Linear(8, 8)

# 在 Sequential 中两次使用同一个 shared 层。
# net[2] 和 net[4] 指向同一个 Linear 对象，因此它们的 weight/bias 是共享的。
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    shared,
    nn.ReLU(),
    shared,
    nn.ReLU(),
    nn.Linear(8, 1)
)

# 执行一次前向传播，确保网络可以正常运行。
net(X)

# 比较 net[2] 和 net[4] 的第一行权重是否相同。
# 因为它们共享参数，所以结果应该全为 True。
print(net[2].weight.data[0] == net[4].weight.data[0])

# 修改 net[2] 的某一个权重。
net[2].weight.data[0, 0] = 100

# 打印 net[2] 的权重矩阵。
print(net[2].weight.data)

# 再次比较：由于 net[2] 和 net[4] 是同一个层，修改 net[2] 也会影响 net[4]。
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([[ 1.0000e+02,  5.4849e-02, -2.4758e-01,  8.7101e-02, -2.9995e-01,
         -8.7463e-02,  1.3358e-01,  4.0351e-02],
        [ 1.5936e-01,  2.9573e-01, -8.0228e-02, -1.9797e-01,  1.2774e-01,
         -1.3191e-01, -3.5099e-01,  8.4589e-02],
        [-8.4767e-02,  6.9338e-02, -3.3492e-01, -8.5376e-02, -3.0877e-01,
          9.8495e-02,  2.3600e-03,  1.2183e-01],
        [-1.5889e-01,  4.3879e-02,  1.5894e-01, -1.7187e-01,  1.2968e-01,
         -1.2442e-01,  1.9221e-01,  2.6171e-02],
        [-2.9786e-01,  4.0009e-02, -3.1724e-01, -2.7500e-01, -2.4931e-01,
          5.2555e-02, -2.3632e-01, -2.8445e-01],
        [-2.3367e-01,  1.2589e-01,  1.0391e-02, -7.7796e-03,  1.1895e-01,
          1.5218e-01,  8.8082e-02,  1.8132e-01],
        [ 7.6469e-02,  3.5512e-02, -2.4411e-01,  2.5666e-01,  7.1597e-02,
         -3.4797e-01,  2.6298e-02, -2.3842e-01],
        [-2.1822e-01,  2.8883e-01, -1.6338e-01, -2.3655e-01, -2.7202e-01,
        

# 11. 自定义层

In [50]:
# 自定义层：和自定义网络一样，本质上也是继承 nn.Module。
# 层可以没有参数，也可以有自己定义的可训练参数。

import torch
import torch.nn.functional as F
from torch import nn

# 1. 构造一个没有任何可训练参数的自定义层。
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        # 对输入张量减去整体均值，使输出张量的均值接近 0。
        return X - X.mean()

# 单独使用 CenteredLayer。
layer = CenteredLayer()
print(layer(torch.FloatTensor([1, 2, 3, 4, 5])))

# 把自定义层放入 Sequential 中，作为复杂模型的一部分。
net = nn.Sequential(
    nn.Linear(8, 128),
    CenteredLayer()
)

# 输入形状为 (4, 8)：4 个样本，每个样本 8 个特征。
Y = net(torch.rand(4, 8))

# 因为最后经过 CenteredLayer，所以输出均值应该非常接近 0。
print(Y.mean())

# 2. 构造一个带可训练参数的自定义层。
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()

        # nn.Parameter 会把普通 Tensor 注册为模型参数。
        # 这样它会出现在 model.parameters() 中，并在训练时被优化器更新。
        # weight 形状为 (输入维度, 输出维度)。
        self.weight = nn.Parameter(torch.randn(in_units, units))

        # bias 形状为 (输出维度,)。
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X):
        # 自己实现线性变换：X @ weight + bias。
        # 这里保留原示例写法使用 .data，便于观察参数值；
        # 真实训练时通常直接使用 self.weight 和 self.bias，以便自动求导完整追踪。
        linear = torch.matmul(X, self.weight.data) + self.bias.data

        # 加入 ReLU 激活函数。
        return F.relu(linear)

# 创建一个自定义线性层：输入 5 维，输出 3 维。
dense = MyLinear(5, 3)

# 打印该层的权重参数。
print(dense.weight)

# 直接把自定义层当作函数使用，输入形状为 (2, 5)，输出形状为 (2, 3)。
print(dense(torch.rand(2, 5)))

# 也可以把自定义层放入 Sequential 中构成更大的模型。
net = nn.Sequential(
    MyLinear(64, 8),
    MyLinear(8, 1)
)

# 输入 2 个样本，每个样本 64 维，最终输出 1 维。
print(net(torch.rand(2, 64)))

tensor([-2., -1.,  0.,  1.,  2.])
tensor(-1.8626e-09, grad_fn=<MeanBackward0>)
Parameter containing:
tensor([[-1.1962,  0.0938, -1.1458],
        [ 0.9532, -0.3662, -1.9544],
        [ 0.2155,  0.0665, -0.6670],
        [ 0.8560, -0.0517,  1.0376],
        [ 0.3615,  0.5815, -1.9913]], requires_grad=True)
tensor([[0.4319, 0.0677, 0.0000],
        [0.0000, 0.1312, 0.0000]])
tensor([[0.],
        [0.]])


# 12. 读写文件

In [51]:
# 读写文件：保存和加载张量、列表、字典。
# 深度学习中经常需要保存中间结果、模型参数或训练检查点。

import torch
from torch import nn
from torch.nn import functional as F

# 1. 保存和加载单个张量。
x = torch.arange(5)  # 得到 tensor([0, 1, 2, 3, 4])

# torch.save(obj, filename) 会把对象序列化保存到文件中。
torch.save(x, 'x-file1')

# torch.load(filename) 会从文件中读回对象。
x2 = torch.load('x-file1')
print(x2)

# 2. 保存和加载张量列表。
y = torch.zeros(4)  # 得到长度为 4 的全 0 张量

torch.save([x, y], 'x-files')

# 读回列表时，可以直接解包成 x2 和 y2。
x2, y2 = torch.load('x-files')
print(x2)
print(y2)

# 3. 保存和加载字典。
# 字典适合保存有名字的多个张量，例如 {'权重': tensor, '偏置': tensor}。
mydict = {'x': x, 'y': y}

torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
print(mydict2)

tensor([0, 1, 2, 3, 4])
tensor([0, 1, 2, 3, 4])
tensor([0., 0., 0., 0.])
{'x': tensor([0, 1, 2, 3, 4]), 'y': tensor([0., 0., 0., 0.])}


/tmp/ipykernel_112337/4289863020.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x2 = torch.load('x-file1')
/tmp/ipykernel_112337/4289863020.py:24: FutureWarning: You a

In [52]:
# 保存和加载模型参数。
# 模型中最重要、最需要保存的是可训练参数，例如 Linear 层的 weight 和 bias。

class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        # hidden：输入 20 维，输出 256 维。
        self.hidden = nn.Linear(20, 256)

        # output：输入 256 维，输出 10 维。
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        # 前向传播：Linear -> ReLU -> Linear。
        return self.output(F.relu(self.hidden(x)))

# 创建一个原始模型。
net = MLP()

# 构造一批输入数据。
X = torch.randn(size=(2, 20))

# 用原始模型计算输出，后面用来和加载后的模型结果比较。
Y = net(X)

# state_dict() 会返回模型所有参数的字典。
# 这里只保存参数，不保存整个模型类定义，因此更灵活、更推荐。
torch.save(net.state_dict(), 'mlp.params')

# 创建一个结构完全相同的新模型。
# 注意：必须先有模型结构，才能把参数加载进去。
clone = MLP()

# load_state_dict 会把文件中的参数复制到 clone 对应的层中。
clone.load_state_dict(torch.load('mlp.params'))

# eval() 表示切换到评估模式。
# 对 Dropout、BatchNorm 等层很重要；本例没有这些层，但养成习惯很好。
print(clone.eval())

# 用加载参数后的模型计算同一个输入。
Y_clone = clone(X)

# 如果参数加载正确，两个模型对同一输入的输出应该完全相同。
print(Y_clone == Y)

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)
tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])


/tmp/ipykernel_112337/245768724.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clone.load_state_dict(torch.load('mlp.params'))
